# 📋 Notebook 1 — Preprocessing & Exploratory Data Analysis
**Dataset:** `engineered_topics2.csv` — 9,055 Dutch parliamentary motions (2009–2026)

**This notebook covers:**
1. Load & inspect the raw dataset
2. Text cleaning for RobBERT input
3. Date / time parsing and validation
4. Numeric feature handling
5. Stratified train / val / test split
6. Rich EDA with visualisations
7. Save `preprocessed.csv`


In [ ]:
# ── Install dependencies (run once) ─────────────────────────────────────────
# Uncomment if not yet installed:
# !pip install pandas numpy scikit-learn matplotlib seaborn plotly


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import re, warnings
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 130, 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.family': 'sans-serif'})
BLUE, RED, GREEN = '#2B5797', '#C0392B', '#27AE60'

print("Libraries loaded ✓")


## 1. Load Dataset

In [ ]:
# ── Adjust path if needed ──────────────────────────────────────────────────
DATA_PATH = "engineered_topics2.csv"   # put the CSV in the same folder as this notebook

df_raw = pd.read_csv(DATA_PATH)
print(f"Shape: {df_raw.shape}")
df_raw.head(3)


In [ ]:
# Basic info
print("=== dtypes ===")
print(df_raw.dtypes)
print("\n=== Null counts ===")
print(df_raw.isnull().sum())
print("\n=== Decision distribution ===")
print(df_raw['Decision'].value_counts())


## 2. Text Cleaning

In [ ]:
def clean_topic(text: str) -> str:
    """
    Minimal NLP pre-cleaning for RobBERT:
      - strip HTML tags
      - remove non-printable / control characters
      - collapse whitespace
    Dutch diacritics (ë, ö, ij …) are kept.
    """
    if not isinstance(text, str):
        return ""
    text = re.sub(r"<[^>]+>", " ", text)                          # HTML tags
    text = re.sub(r"[^\x20-\x7E\u00C0-\u024F\u0300-\u036F]", " ", text)  # keep Latin+diacritics
    text = re.sub(r"\s+", " ", text).strip()
    return text

df = df_raw.copy()
df['Topic_clean'] = df['Topic'].apply(clean_topic)

# Show a few examples
sample = df[['Topic', 'Topic_clean']].sample(4, random_state=42)
for _, row in sample.iterrows():
    print(f"ORIGINAL : {row['Topic'][:100]}")
    print(f"CLEANED  : {row['Topic_clean'][:100]}")
    print()


In [ ]:
# Character length distribution — what actually matters for RobBERT
# RobBERT's tokeniser splits on subword units; character length is the
# correct measure of how much text each title contains.
# Rule of thumb: ~4-5 chars per RobBERT token for Dutch text.
ROBBERT_MAX_CHARS = 128 * 4.5   # ≈ 576 chars → conservative estimate

df['text_length'] = df['Topic_clean'].str.len()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# ── Left: full distribution ───────────────────────────────────────────────
ax = axes[0]
ax.hist(df['text_length'], bins=60, color=BLUE, edgecolor='white', alpha=0.85)
ax.axvline(ROBBERT_MAX_CHARS, color=RED, linestyle='--', linewidth=1.5,
           label=f'RobBERT ~512 char limit')
ax.set_xlabel('Text Length (characters)')
ax.set_ylabel('Number of Motions')
ax.set_title('Distribution of Motion Text Length (characters)', fontweight='bold')
ax.legend()

# Annotate percentiles
for pct, color in [(50, GREEN), (75, ORANGE), (95, RED)]:
    val = df['text_length'].quantile(pct/100)
    ax.axvline(val, color=color, linestyle=':', linewidth=1.2,
               label=f'p{pct} = {val:.0f}')
ax.legend(fontsize=8)

# ── Right: zoomed tail (long titles) ─────────────────────────────────────
ax2 = axes[1]
tail = df[df['text_length'] > 200]['text_length']
ax2.hist(tail, bins=40, color='#E67E22', edgecolor='white', alpha=0.85)
ax2.axvline(ROBBERT_MAX_CHARS, color=RED, linestyle='--', linewidth=1.5,
            label=f'~512 char limit')
ax2.set_xlabel('Text Length (characters)')
ax2.set_ylabel('Number of Motions')
ax2.set_title('Tail: Motions with >200 Characters', fontweight='bold')
ax2.legend(fontsize=8)

plt.tight_layout(); plt.show()

# Summary stats
print("=== Text Length Statistics ===")
print(df['text_length'].describe().round(1).to_string())
print(f"\nMedian:           {df['text_length'].median():.0f} chars")
print(f"p75:              {df['text_length'].quantile(0.75):.0f} chars")
print(f"p95:              {df['text_length'].quantile(0.95):.0f} chars")
pct_over = (df['text_length'] > ROBBERT_MAX_CHARS).mean() * 100
print(f"\nTitles likely exceeding RobBERT's limit (~576 chars): {pct_over:.1f}%")
print("→ For precise truncation counts, use the actual RobBERT tokeniser (see below)")


## 3. Date / Time Parsing

In [ ]:
df['Topic_date'] = pd.to_datetime(df['Topic_date'], errors='coerce')
df['year']        = df['Topic_date'].dt.year.astype('Int64')
df['month']       = df['Topic_date'].dt.month.astype('Int64')
df['day_of_week'] = df['Topic_date'].dt.dayofweek.astype('Int64')  # 0=Mon

def parse_hour(t):
    try: return int(str(t).split(':')[0])
    except: return -1

df['hour'] = df['Time_of_day_time'].apply(parse_hour)

# Recalculate Time_of_day_category where missing
def hour_to_tod(h):
    if h < 0:   return 'Unknown'
    if h < 9:   return 'Early morning'
    if h < 12:  return 'Late morning'
    if h < 15:  return 'Early afternoon'
    if h < 18:  return 'Late afternoon'
    if h < 22:  return 'Evening'
    return 'Night'

mask = df['Time_of_day_category'].isna() | (df['Time_of_day_category'] == '')
df.loc[mask, 'Time_of_day_category'] = df.loc[mask, 'hour'].apply(hour_to_tod)

print("Date/time columns parsed:")
print(df[['Topic_date','year','month','day_of_week','hour','Time_of_day_category']].head(5))


## 4. Numeric Cleaning

In [ ]:
# Duration — clip at 99th percentile, fill NaN with median
df['Topic_duration_minutes'] = pd.to_numeric(df['Topic_duration_minutes'], errors='coerce')
p99 = df['Topic_duration_minutes'].quantile(0.99)
df['Topic_duration_minutes'] = df['Topic_duration_minutes'].clip(upper=p99)
med_dur = df['Topic_duration_minutes'].median()
df['Topic_duration_minutes'] = df['Topic_duration_minutes'].fillna(med_dur)
print(f"Duration — clipped at {p99:.0f} min | NaN filled with median {med_dur:.0f} min")

# Density — fill NaN
df['Topic_density_per_day'] = pd.to_numeric(df['Topic_density_per_day'], errors='coerce')
df['Topic_density_per_day'] = df['Topic_density_per_day'].fillna(df['Topic_density_per_day'].median())

# Binary label
df['label'] = (df['Decision'] == 'Accepted').astype(int)

print("\nNull check after cleaning:")
print(df[['Topic_duration_minutes','Topic_density_per_day','label']].isnull().sum())


## 5. Train / Val / Test Split

In [ ]:
# Stratified 70 / 15 / 15
ids = df['Id']
y   = df['label']

ids_train, ids_temp, _, y_temp = train_test_split(
    ids, y, test_size=0.30, random_state=42, stratify=y)
ids_val, ids_test = train_test_split(
    ids_temp, test_size=0.50, random_state=42, stratify=y_temp)

df['split'] = 'train'
df.loc[df['Id'].isin(ids_val.values),  'split'] = 'val'
df.loc[df['Id'].isin(ids_test.values), 'split'] = 'test'

for split, grp in df.groupby('split'):
    n = len(grp)
    acc = grp['label'].mean()*100
    print(f"  {split:6s}: {n:5d} rows  |  {acc:.1f}% Accepted")


## 6. Exploratory Data Analysis

In [ ]:
# ── 6a. Acceptance rate by Time of Day ───────────────────────────────────────
tod_order = ['Early morning','Late morning','Early afternoon','Late afternoon','Evening','Night']
tod = (df.groupby('Time_of_day_category')['label']
         .agg(['mean','count'])
         .reindex(tod_order)
         .rename(columns={'mean':'acc_rate','count':'n'}))

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(tod.index, tod['acc_rate']*100,
              color=[GREEN if v>50 else RED for v in tod['acc_rate']],
              edgecolor='white', width=0.6, alpha=0.88)
ax.axhline(50, color='grey', linestyle='--', linewidth=0.8)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_ylim(0, 75)
ax.set_xlabel('Time of Day', fontsize=11)
ax.set_ylabel('Acceptance Rate', fontsize=11)
ax.set_title('Motion Acceptance Rate by Time of Day', fontsize=13, fontweight='bold')
for bar, n in zip(bars, tod['n']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            f'n={n}', ha='center', va='bottom', fontsize=8, color='#555')
plt.xticks(rotation=20, ha='right')
plt.tight_layout(); plt.show()

print(tod.to_string())


In [ ]:
# ── 6b. Acceptance rate by Meeting Type ──────────────────────────────────────
mt = df.groupby('Meeting_type')['label'].agg(['mean','count']).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

ax = axes[0]
ax.bar(mt['Meeting_type'], mt['mean']*100,
       color=[BLUE, '#E67E22'], edgecolor='white', width=0.5)
ax.axhline(50, color='grey', linestyle='--', linewidth=0.8)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_title('Acceptance by Meeting Type', fontweight='bold')
ax.set_ylim(0, 75)
for i, row in mt.iterrows():
    ax.text(i, row['mean']*100+0.5, f"{row['mean']*100:.1f}%\n(n={row['count']})",
            ha='center', va='bottom', fontsize=9)

# ── 6c. Motions per year ────────────────────────────────────────────────────
ax2 = axes[1]
yr = df.groupby(['year','Decision'])['Id'].count().unstack(fill_value=0)
yr.plot(kind='bar', stacked=True, ax=ax2,
        color=[RED, GREEN], edgecolor='white', alpha=0.85)
ax2.set_title('Motions per Year by Outcome', fontweight='bold')
ax2.set_xlabel('Year'); ax2.set_ylabel('Count')
ax2.tick_params(axis='x', rotation=45)
ax2.legend(loc='upper left')

plt.tight_layout(); plt.show()


In [ ]:
# ── 6d. Acceptance by Topic Category ─────────────────────────────────────────
cat = (df.groupby('Topic_category')['label']
         .agg(['mean','count'])
         .sort_values('mean', ascending=True)
         .rename(columns={'mean':'acc','count':'n'}))

fig, ax = plt.subplots(figsize=(9, 5))
colors = [GREEN if v > 0.5 else RED for v in cat['acc']]
h = ax.barh(cat.index, cat['acc']*100, color=colors, edgecolor='white', alpha=0.85)
ax.axvline(50, color='grey', linestyle='--', linewidth=0.8)
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_xlabel('Acceptance Rate')
ax.set_title('Acceptance Rate by Topic Category', fontsize=13, fontweight='bold')
for bar, n in zip(h, cat['n']):
    ax.text(bar.get_width()+0.3, bar.get_y()+bar.get_height()/2,
            f'  n={n}', va='center', fontsize=8, color='#555')
plt.tight_layout(); plt.show()


In [ ]:
# ── 6e. Heatmap: hour × day-of-week acceptance rate ──────────────────────────
df_valid = df[(df['hour'] >= 0) & df['day_of_week'].notna()].copy()
hm = (df_valid.groupby(['hour','day_of_week'])['label']
               .mean()
               .unstack(fill_value=np.nan))
hm.columns = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun'][:len(hm.columns)]

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(hm, ax=ax, cmap='RdYlGn', vmin=0, vmax=1,
            linewidths=0.4, linecolor='white',
            cbar_kws={'label': 'Acceptance Rate', 'format': '%.0%%'})
ax.set_title('Acceptance Rate: Hour of Day × Day of Week', fontsize=13, fontweight='bold')
ax.set_xlabel('Day of Week'); ax.set_ylabel('Hour of Day')
plt.tight_layout(); plt.show()


In [ ]:
# ── 6f. Duration distribution by outcome ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 3))
for decision, color, label in [('Accepted', GREEN, 'Accepted'),
                                 ('Rejected', RED, 'Rejected')]:
    vals = df[df['Decision'] == decision]['Topic_duration_minutes']
    ax.hist(vals, bins=50, alpha=0.6, color=color, label=label, edgecolor='white')
ax.set_xlabel('Session Duration (minutes)')
ax.set_ylabel('Count')
ax.set_title('Session Duration by Outcome', fontweight='bold')
ax.legend()
plt.tight_layout(); plt.show()

print("Median duration (Accepted):", df[df['label']==1]['Topic_duration_minutes'].median())
print("Median duration (Rejected):", df[df['label']==0]['Topic_duration_minutes'].median())


## 7. Save Preprocessed Data

In [ ]:
OUTPUT_PATH = "preprocessed.csv"
df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(df)} rows → {OUTPUT_PATH}")
print(f"Columns: {df.columns.tolist()}")
